# Prediksi Return USD/IDR dari Berita Geopolitik (NLP)

Pipeline: cleaning berita & kurs BI → skor sentimen (VADER pada judul, Loughran-McDonald pada isi) → alignment ke hari bursa → agregasi harian → regresi return.

In [1]:
import os
import re
import html
import unicodedata
import warnings
from collections import Counter
from datetime import time

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [2]:
news_raw = pd.read_csv('data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [3]:
news_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16733 entries, 0 to 16732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   keyword_matched     16733 non-null  object 
 1   title               16733 non-null  object 
 2   url                 16733 non-null  object 
 3   preview_summary     16733 non-null  object 
 4   full_text           13431 non-null  object 
 5   date_raw            16733 non-null  object 
 6   section             16649 non-null  object 
 7   published_raw       15239 non-null  object 
 8   published_date      15239 non-null  object 
 9   published_time      15239 non-null  object 
 10  published_timezone  15239 non-null  float64
 11  published_iso       15239 non-null  object 
 12  scrape_status       16733 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.7+ MB


In [4]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [5]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning
## Filter Dataset

In [6]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


In [7]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


In [8]:
# Panjang text minimal 350

effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 4349


In [9]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 4349
Setelah filter kategori/section     : 4336


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [10]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 4336
Sisa NaN pada full_text: 0


## 2. Data News Cleaning

In [11]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4336


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


In [12]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 54


,simbol,nama_unicode,jumlah
0,.,FULL STOP,232989
1,",",COMMA,203423
2,"""",QUOTATION MARK,88912
3,',APOSTROPHE,60351
4,-,HYPHEN-MINUS,41818
5,%,PERCENT SIGN,18915
6,—,EM DASH,10379
7,$,DOLLAR SIGN,9806
8,’,RIGHT SINGLE QUOTATION MARK,6782
9,:,COLON,5100


In [13]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


In [14]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


In [15]:
#memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
en    4336
Name: count, dtype: int64

Setelah filter bahasa Inggris (en) : 4336 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


## 3. Cleaning Data Kurs

In [16]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [17]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [18]:
#stopword removal
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "also", "said", "says", "say",
    "would", "could", "also", "one", "two", "get", "got",
}
STOPWORDS_EN = set(stopwords.words("english")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Inggris & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_EN and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 862.4
Rata-rata jumlah kata sesudah stopword removal  : 494.5


,lower_text,text_no_stopwords
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...


In [19]:
#lemmatization (POS aware)
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_tokens(tokens: list) -> list:
    # Lemmatisasi tiap token berdasarkan POS tag-nya (POS-aware lemmatization).
    if not tokens:
        return []
    tagged_tokens = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(token, pos=get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
2,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...


# Data Labelling
## VADER

In [20]:
# ===== VADER: skor sentimen JUDUL =====
USE_VADER = True      # True = aktif, False = nonaktif
THR = 0.05            # ambang standar VADER untuk compound

from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)
vader = SentimentIntensityAnalyzer()

VADER_COLS = ['vader_title', 'vader_title_pos', 'vader_title_neg', 'vader_title_neu']

def score_title(t):
    # judul kosong -> semua NaN (bukan 0 / netral)
    if isinstance(t, str) and t.strip():
        s = vader.polarity_scores(t)                      # dipanggil sekali per judul
        return [s['compound'], s['pos'], s['neg'], s['neu']]
    return [np.nan] * 4

# bersihkan sisa run sebelumnya
news = news.drop(columns=[c for c in news.columns if c.startswith('vader_title')])

if USE_VADER:
    # pakai title ASLI (bukan lower_text / lemma): VADER membaca huruf kapital, tanda seru, dan negasi
    v = pd.DataFrame(news['title'].apply(score_title).tolist(), columns=VADER_COLS, index=news.index)
    news = pd.concat([news, v], axis=1)

    # polaritas dari compound: > 0.05 -> +1, < -0.05 -> -1, sisanya 0. NaN tetap NaN.
    comp = news['vader_title']
    pol = np.select([comp > THR, comp < -THR], [1, -1], default=0).astype(float)
    news['vader_title_polarity'] = pd.Series(pol, index=news.index).where(comp.notna())

    valid = news['vader_title'].dropna()
    print('VADER aktif')
    print(f"Total: {len(news)} | NaN: {news['vader_title'].isna().sum()} -> tetap NaN")
    print("Distribusi polaritas (dari judul valid):")
    print(news['vader_title_polarity'].value_counts(normalize=True).sort_index().round(3).to_string())
    print(f"Rata-rata proporsi kata | pos: {news['vader_title_pos'].mean():.3f} | "
          f"neg: {news['vader_title_neg'].mean():.3f} | neu: {news['vader_title_neu'].mean():.3f}")
    display(news[['title'] + VADER_COLS + ['vader_title_polarity']].head())
else:
    print('VADER dinonaktifkan')

VADER aktif
Total: 4336 | NaN: 0 -> tetap NaN
Distribusi polaritas (dari judul valid):
vader_title_polarity
-1.0    0.444
 0.0    0.293
 1.0    0.263
Rata-rata proporsi kata | pos: 0.076 | neg: 0.123 | neu: 0.800


,title,vader_title,vader_title_pos,vader_title_neg,vader_title_neu,vader_title_polarity
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.0000,0.000,0.000,1.000,0.0
1,Oil rises 2% as White House says no US-Iran talks happening,-0.2960,0.000,0.180,0.820,-1.0
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,-0.5106,0.000,0.235,0.765,-1.0
3,China’s super-rich fled Singapore. Now they want to come back,0.0772,0.126,0.000,0.874,1.0
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,-0.2732,0.000,0.130,0.870,-1.0


In [21]:
#tampilkan published_date yang NaN
news[news['published_date'].isna()]

#drop baris yang published_date nya NaN
news = news.dropna(subset=['published_date']).reset_index(drop=True)

In [22]:
# ===== SENTIMEN BERBASIS ASPEK: berita tentang AS vs Indonesia (per artikel) =====
FT_MIN_US = 5    # artikel dianggap terkait AS jika ada di judul/ringkasan, ATAU istilahnya muncul >= 5x di isi
FT_MIN_ID = 2    # sama untuk Indonesia (ambang lebih rendah karena istilahnya jauh lebih jarang)

# Skor per artikel yang dipecah per aspek (hanya kolom yang sudah ada di news)
ASPECT_SCORES = [c for c in ['vader_title', 'lm_text_net', 'geo_risk_index'] if c in news.columns]

# Istilah huruf-kecil dicocokkan tanpa peduli kapital; 'US', 'U.S.', 'Fed', 'BI', 'IDR' harus persis
US_RE = re.compile(
    r"(?i:\b(?:united states|america|american|americans|washington|white house|federal reserve|"
    r"treasury|greenback|dollar|congress|pentagon|wall street|trump|biden|powell)\b)"
    r"|\bU\.S\.A?|\bUSA\b|\bUS\b|\bFed\b")
ID_RE = re.compile(
    r"(?i:\b(?:indonesia|indonesian|indonesians|jakarta|rupiah|bank indonesia|prabowo|jokowi|widodo|"
    r"sri mulyani|purbaya|danantara|perry warjiyo)\b)"
    r"|\bIDR\b|\bIHSG\b|\bOJK\b|\bBI\b")

# Teks mentah (bukan lower_text): huruf besar kecil dibutuhkan regex di atas
short_txt = news['title'].fillna('') + ' ' + news['preview_summary'].fillna('')
long_txt  = news['full_text'].fillna('')

def has_aspect(short, long_, rx, ft_min):
    return int(len(rx.findall(short)) >= 1 or len(rx.findall(long_)) >= ft_min)

news['is_us'] = [has_aspect(s, l, US_RE, FT_MIN_US) for s, l in zip(short_txt, long_txt)]
news['is_id'] = [has_aspect(s, l, ID_RE, FT_MIN_ID) for s, l in zip(short_txt, long_txt)]

# Skor per aspek: nilai skor jika artikel terkait aspek itu, selain itu NaN
news = news.drop(columns=[c for c in news.columns
                          if c.endswith(('_us', '_id')) and c.startswith(('vader_title', 'lm_text_', 'geo_'))])
ASPECT_ARTICLE_COLS = []
for sc in ASPECT_SCORES:
    for asp in ('us', 'id'):
        col = f'{sc}_{asp}'
        news[col] = news[sc].where(news[f'is_{asp}'] == 1)
        ASPECT_ARTICLE_COLS.append(col)

# Nama kolom setelah agregasi harian (dipakai nanti di cell agregasi/merge)
ASPECT_DAILY_COLS = [c + '_mean' if c.startswith('vader_') else c for c in ASPECT_ARTICLE_COLS]

# ---------- Cek per artikel (tanpa agregasi, tanpa penyelarasan tanggal) ----------
both  = ((news['is_us'] == 1) & (news['is_id'] == 1)).sum()
only_us = ((news['is_us'] == 1) & (news['is_id'] == 0)).sum()
only_id = ((news['is_us'] == 0) & (news['is_id'] == 1)).sum()
none_  = ((news['is_us'] == 0) & (news['is_id'] == 0)).sum()
print(f"Total artikel: {len(news)}")
print(f"Terkait AS: {news['is_us'].sum()} ({news['is_us'].mean():.1%}) | "
      f"terkait Indonesia: {news['is_id'].sum()} ({news['is_id'].mean():.1%})")
print(f"Hanya AS: {only_us} | hanya Indonesia: {only_id} | keduanya: {both} | tidak keduanya: {none_}")
print("Skor yang dipecah:", ASPECT_SCORES if ASPECT_SCORES else "TIDAK ADA (jalankan cell VADER/LM/geopolitik dulu)")

# Periksa kualitas penandaan lewat contoh judul
print("\nContoh judul ditandai Indonesia:")
display(news.loc[news['is_id'] == 1, ['title', 'is_us', 'is_id']].head(10))
print("Contoh judul yang TIDAK ditandai AS maupun Indonesia:")
display(news.loc[(news['is_us'] == 0) & (news['is_id'] == 0), ['title']].head(10))

Total artikel: 4335
Terkait AS: 2963 (68.4%) | terkait Indonesia: 39 (0.9%)
Hanya AS: 2939 | hanya Indonesia: 15 | keduanya: 24 | tidak keduanya: 1357
Skor yang dipecah: ['vader_title']

Contoh judul ditandai Indonesia:


,title,is_us,is_id
173,Investors are looking to this ‘strategic base’ for refuge,0,1
335,Southeast Asia is a top choice for firms diversifying supply chains amid U.S.-China tensions,1,1
376,"U.S. inflation relief dents dollar, yen gains ahead of BOJ",1,1
395,Here are the top concerns of ultra-rich investors in Asia-Pacific,0,1
399,U.S. shifts focus to China risks as Zelenskyy arrives at Asia defense meeting,1,1
454,"Here are 3 hottest markets in Southeast Asia for 2022, according to Wall Street",1,1
511,"U.S. loses its spot to China as Southeast Asia’s most favored ally, survey shows",1,1
518,Jokowi says rising food prices are dangerous and put developing nations like Indonesia atrisk,1,1
541,"U.S. and China vie for dominance in new ‘a la carte’ world order, survey finds",1,1
691,A global food price shock looms as Middle East war rages on. Here’s who will ...,0,1


Contoh judul yang TIDAK ditandai AS maupun Indonesia:


,title
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise
3,China’s super-rich fled Singapore. Now they want to come back
5,"Market correctionrisklooks elevated as stocks hit record highs, top Europe central banker warns"
7,"Japan’s megabanks post record profits, but analysts warn growth may slow as risks mount"
11,"Arm co-founder Hermann Hauser’s AI warning: The revolution is real, but so is the bubble ..."
18,"Another day another high: Gold surges past $5,100 as investors seek shelter from global risks"
21,Jamie Dimon saysgeopoliticalrisks are surging: ‘Conditions are treacherous and getting worse’
25,"Markets underestimategeopoliticalriskas raft of elections looms, ECB’s De Guindos says"
41,How the ‘double scar’ of past inflation woes andgeopoliticalshocks amid the Iran war ...
49,"Conflict, extreme weather and disinformation top global risks in 2025, Davos survey says"


In [23]:
# 1. Buang kolom aspek terpisah (sumber NaN sebelumnya)
news = news.drop(columns=[c for c in news.columns
                          if c.endswith(('_us', '_id')) and c.startswith(('vader_title', 'lm_text_', 'geo_'))])

# 2. Simpan skor asli sekali; perhitungan selalu dari skor asli, jadi aman kalau cell dijalankan berulang
SCORE_COLS = ['vader_title', 'vader_title_polarity', 'vader_title_pos', 'vader_title_neg']
for c in SCORE_COLS:
    if f'{c}_raw' not in news.columns:
        news[f'{c}_raw'] = news[c]

# 3. Balik hanya berita yang terkait Indonesia saja (berita yang menyebut AS juga, atau tidak keduanya, tetap)
flip = (news['is_id'] == 1) & (news['is_us'] == 0)
sign = np.where(flip, -1.0, 1.0)

news['vader_title']          = news['vader_title_raw'] * sign + 0.0            # NaN tetap NaN; +0.0 mencegah -0.0
news['vader_title_polarity'] = news['vader_title_polarity_raw'] * sign + 0.0
# pos dan neg ditukar pada baris yang dibalik, supaya konsisten dengan compound (neu tidak berubah)
news['vader_title_pos'] = np.where(flip, news['vader_title_neg_raw'], news['vader_title_pos_raw'])
news['vader_title_neg'] = np.where(flip, news['vader_title_pos_raw'], news['vader_title_neg_raw'])

# 4. Cek
print(f"Artikel dibalik: {flip.sum()} dari {len(news)} ({flip.mean():.1%})")
print(f"NaN vader_title: {news['vader_title'].isna().sum()} (harus sama dengan jumlah judul kosong)")
display(news.loc[flip, ['title', 'vader_title_raw', 'vader_title',
                        'vader_title_polarity_raw', 'vader_title_polarity']].head(10))

Artikel dibalik: 15 dari 4335 (0.3%)
NaN vader_title: 0 (harus sama dengan jumlah judul kosong)


,title,vader_title_raw,vader_title,vader_title_polarity_raw,vader_title_polarity
173,Investors are looking to this ‘strategic base’ for refuge,0.0000,0.0000,0.0,0.0
395,Here are the top concerns of ultra-rich investors in Asia-Pacific,0.2023,-0.2023,1.0,-1.0
691,A global food price shock looms as Middle East war rages on. Here’s who will ...,-0.8807,0.8807,-1.0,1.0
740,We spoke to over 30 CEOs and business leaders. Here’s what worries them most,-0.4215,0.4215,-1.0,1.0
1232,"Over $1 trillion needed for developing nations’ climate transition, says ex-World Bank official",0.0000,0.0000,0.0,0.0
1555,Indonesia is the best-performing Asia-Pacific market so far this year,0.0000,0.0000,0.0,0.0
1785,"Fearful of the ‘middle-income trap,’ Indonesia pushes for major policy reform",-0.4939,0.4939,-1.0,1.0
1823,"Five countries, other than China, most dependent on the South China Sea",0.0000,0.0000,0.0,0.0
2382,Shares of Tencent-backed J&T Express fall in lackluster Hong Kong debut,0.2960,-0.2960,1.0,-1.0
2531,Aglobalfood price shock looms as Middle East war rages on. Here’s who will ...,-0.8807,0.8807,-1.0,1.0


## Loughran-McDonald

In [24]:
# ===== Loughran-McDonald pada lower_text =====
LM_PATH = 'data/raw/Loughran-McDonald_MasterDictionary_1993-2025.csv'   # sesuaikan nama file

lm = pd.read_csv(LM_PATH)
lm.columns = [c.strip() for c in lm.columns]
lm['Word'] = lm['Word'].astype(str).str.lower()

# cocokkan nama kolom tanpa peduli underscore/kapital, jadi tahan terhadap beda versi file
norm = {c.lower().replace('_', '').replace(' ', ''): c for c in lm.columns}
LM_CATS = {'neg': 'negative', 'pos': 'positive', 'unc': 'uncertainty', 'lit': 'litigious',
           'strong': 'strongmodal', 'weak': 'weakmodal', 'constr': 'constraining'}
missing = [v for v in LM_CATS.values() if v not in norm]
assert not missing, f"kolom tidak ditemukan: {missing}. Kolom tersedia: {list(lm.columns)}"

# kolom kategori bernilai 0 jika bukan anggota, selain itu berisi tahun kata ditambahkan
LM_SETS = {k: frozenset(lm.loc[lm[norm[v]] > 0, 'Word']) for k, v in LM_CATS.items()}
print({k: len(v) for k, v in LM_SETS.items()})

def lm_score(text):
    toks = text.split()
    n = len(toks)
    if n == 0:
        return {'lm_text_n_tokens': 0, **{f'lm_text_{k}': np.nan for k in LM_SETS},
                'lm_text_net': np.nan, 'lm_text_polarity': np.nan}
    cnt = {k: sum(t in s for t in toks) for k, s in LM_SETS.items()}
    p, ng = cnt['pos'], cnt['neg']
    out = {'lm_text_n_tokens': n}
    out.update({f'lm_text_{k}': v / n for k, v in cnt.items()})            # proporsi terhadap total token
    out['lm_text_net'] = (p - ng) / n                                       # (pos - neg) / total token
    out['lm_text_polarity'] = (p - ng) / (p + ng) if (p + ng) > 0 else 0.0  # skala -1..1
    return out

scores = pd.DataFrame([lm_score(t) for t in news['lower_text']], index=news.index)
news = news.drop(columns=[c for c in scores.columns if c in news.columns]).join(scores)   # aman dijalankan ulang

# ---------- Cek ----------
LM_COLS = list(scores.columns)
print(f"Artikel: {len(news)} | lower_text kosong: {(news['lm_text_n_tokens'] == 0).sum()}")
display(news[LM_COLS].describe().T[['mean', 'std', 'min', '50%', 'max']])

if 'vader_title' in news.columns:
    print("Korelasi lm_text_net vs vader_title:", round(news['lm_text_net'].corr(news['vader_title']), 3))

# kata LM paling sering muncul (sanity check: harus masuk akal untuk berita geopolitik-ekonomi)
tok_counter = Counter(t for txt in news['lower_text'] for t in txt.split())
for k in ('neg', 'pos', 'unc'):
    top = [(w, c) for w, c in tok_counter.most_common() if w in LM_SETS[k]][:12]
    print(f"\nTop {k}:", top)

print("\nArtikel paling negatif:")
display(news.nsmallest(5, 'lm_text_net')[['title', 'lm_text_net']])
print("Artikel paling positif:")
display(news.nlargest(5, 'lm_text_net')[['title', 'lm_text_net']])

{'neg': 2345, 'pos': 347, 'unc': 297, 'lit': 903, 'strong': 19, 'weak': 27, 'constr': 184}
Artikel: 4335 | lower_text kosong: 0


,mean,std,min,50%,max
lm_text_n_tokens,862.495732,426.637639,396.000000,775.000000,12435.000000
lm_text_neg,0.022857,0.010865,0.000000,0.021711,0.074394
lm_text_pos,0.007417,0.005132,0.000000,0.006467,0.037037
lm_text_unc,0.009524,0.006011,0.000000,0.008446,0.042654
lm_text_lit,0.002581,0.005153,0.000000,0.001213,0.086498
lm_text_strong,0.004405,0.003524,0.000000,0.003683,0.025581
lm_text_weak,0.004765,0.003598,0.000000,0.004152,0.035545
lm_text_constr,0.001912,0.002280,0.000000,0.001384,0.024869
lm_text_net,-0.015440,0.013053,-0.072664,-0.014717,0.036000
lm_text_polarity,-0.475604,0.351321,-1.000000,-0.534884,1.000000


Korelasi lm_text_net vs vader_title: 0.369

Top neg: [('conflict', 4088), ('against', 3876), ('concerns', 1851), ('cut', 1487), ('crisis', 1366), ('warned', 1153), ('threat', 919), ('fears', 842), ('volatility', 718), ('losses', 663), ('disruption', 655), ('recession', 633)]

Top pos: [('despite', 1514), ('strong', 1298), ('good', 1115), ('better', 904), ('gains', 903), ('able', 771), ('highest', 752), ('best', 733), ('leading', 663), ('boost', 636), ('leadership', 611), ('alliance', 610)]

Top unc: [('could', 7151), ('may', 3738), ('risk', 2959), ('risks', 1681), ('nearly', 1353), ('uncertainty', 1067), ('might', 981), ('possible', 953), ('almost', 819), ('volatility', 718), ('believe', 696), ('roughly', 628)]

Artikel paling negatif:


,title,lm_text_net
2719,Defense Department warns climate change will increase conflicts over water and food,-0.072664
2610,Oil prices climb more than $1 on fear of spreading Middle Eastconflict,-0.068337
1230,"‘Aggressive’ China cyberattacks are the ‘defining threat’ of our time, top U.S. cyber official says",-0.067416
2385,"Binance CEO Changpeng Zhao pleads guilty to federal charges, steps down",-0.066510
2205,Harvard professors warn that war-torn countries will miss global vaccine goals in 2022,-0.061798


Artikel paling positif:


,title,lm_text_net
1405,"Here are the top 10 U.S. stock analysts, according to TipRanks",0.036000
1314,"Geopoliticaltensionswith the U.S. could ‘supercharge’ China’s innovation, JPMorgan says",0.030501
1503,Top Wall Street analysts favor these stocks for attractive long-term potential,0.028302
3225,Top Wall Street analysts prefer these dividend stocks for boosting portfolio returns,0.026911
1519,Top Wall Street analysts are confident about the potential behind these 3 stocks,0.025826


# Penyelerasan Data & Agregasi Harian

Alur bagian ini:
1. **Flip LM** untuk artikel Indonesia-only (aturan sama dengan VADER).
2. **Alignment**: artikel yang terbit setelah cutoff **15:00 WIB** digeser ke hari bursa berikutnya; weekend dan libur nasional otomatis ikut ke hari bursa BI berikutnya (`merge_asof` ke kalender BI aktual).
3. **Agregasi per hari bursa**: skor dipool dari jumlah (bukan rata-rata rata-rata), volume berita disimpan sebagai fitur tersendiri, sehingga bobot skor tidak hilang saat banyak artikel tergabung di satu hari.
4. **Bobot waktu** (ablation) dan teks harian untuk TF-IDF (dari teks asli, bukan lemmatized).

In [25]:
# ===== Flip LM untuk artikel Indonesia-only (aturan sama dengan VADER) =====
LM_FLIP_COLS = ['lm_text_pos', 'lm_text_neg', 'lm_text_net', 'lm_text_polarity']
for c in LM_FLIP_COLS:
    if f'{c}_raw' not in news.columns:          # simpan skor asli sekali, aman dijalankan ulang
        news[f'{c}_raw'] = news[c]

flip = (news['is_id'] == 1) & (news['is_us'] == 0)
sign = np.where(flip, -1.0, 1.0)

news['lm_text_net']      = news['lm_text_net_raw'] * sign + 0.0
news['lm_text_polarity'] = news['lm_text_polarity_raw'] * sign + 0.0
# pos dan neg ditukar pada baris yang dibalik; unc/lit/modal tidak diubah
news['lm_text_pos'] = np.where(flip, news['lm_text_neg_raw'], news['lm_text_pos_raw'])
news['lm_text_neg'] = np.where(flip, news['lm_text_pos_raw'], news['lm_text_neg_raw'])

assert np.allclose(news['lm_text_pos'] - news['lm_text_neg'], news['lm_text_net']), "pos/neg/net tidak konsisten"
print(f"Artikel dibalik: {flip.sum()} dari {len(news)} ({flip.mean():.1%})")
display(news.loc[flip, ['title', 'vader_title_raw', 'vader_title', 'lm_text_net_raw', 'lm_text_net']].head(8))

Artikel dibalik: 15 dari 4335 (0.3%)


,title,vader_title_raw,vader_title,lm_text_net_raw,lm_text_net
173,Investors are looking to this ‘strategic base’ for refuge,0.0000,0.0000,0.006130,-0.006130
395,Here are the top concerns of ultra-rich investors in Asia-Pacific,0.2023,-0.2023,-0.028169,0.028169
691,A global food price shock looms as Middle East war rages on. Here’s who will ...,-0.8807,0.8807,-0.023175,0.023175
740,We spoke to over 30 CEOs and business leaders. Here’s what worries them most,-0.4215,0.4215,-0.025180,0.025180
1232,"Over $1 trillion needed for developing nations’ climate transition, says ex-World Bank official",0.0000,0.0000,-0.020927,0.020927
1555,Indonesia is the best-performing Asia-Pacific market so far this year,0.0000,0.0000,0.002857,-0.002857
1785,"Fearful of the ‘middle-income trap,’ Indonesia pushes for major policy reform",-0.4939,0.4939,0.012422,-0.012422
1823,"Five countries, other than China, most dependent on the South China Sea",0.0000,0.0000,-0.010417,0.010417


In [26]:
# ===== Alignment ke hari bursa BI =====
CUTOFF = time(15, 0)     # setelah 15:00 WIB -> hari bursa berikutnya (tepat 15:00:00 masih hari yang sama)

# timestamp kanonik: published_raw -> WIB (naif). Dipakai untuk alignment, bobot waktu, dan cek lain.
news['ts_wib'] = (pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
                    .dt.tz_convert('Asia/Jakarta').dt.tz_localize(None))
n_nat = int(news['ts_wib'].isna().sum())
news = news.dropna(subset=['ts_wib']).reset_index(drop=True)

after = news['ts_wib'].dt.time.map(lambda t: t > CUTOFF)
news['cand_date'] = (news['ts_wib'].dt.normalize() + pd.to_timedelta(after.astype(int), unit='D')).astype('datetime64[ns]')
news['shifted'] = after.astype(int)

# lompat ke tanggal BI terdekat setelahnya -> weekend & libur nasional otomatis tergabung ke hari bursa berikutnya
bi_dates = (bi[['Tanggal_dt']].rename(columns={'Tanggal_dt': 'aligned_trading_date'})
              .astype({'aligned_trading_date': 'datetime64[ns]'}).sort_values('aligned_trading_date'))

news = news.drop(columns=['aligned_trading_date'], errors='ignore')
news = pd.merge_asof(news.sort_values('cand_date'), bi_dates,
                     left_on='cand_date', right_on='aligned_trading_date', direction='forward')

n_out = int(news['aligned_trading_date'].isna().sum())
news = news.dropna(subset=['aligned_trading_date']).sort_values('ts_wib').reset_index(drop=True)

# ---------- Cek ----------
print(f"ts_wib NaT (dibuang): {n_nat} | di luar rentang BI (dibuang): {n_out} | artikel tersisa: {len(news)}")
print(f"Digeser karena lewat {CUTOFF}: {news['shifted'].mean():.1%}")
print("Hari aligned (0=Senin):", news['aligned_trading_date'].dt.weekday.value_counts().sort_index().to_dict())
print("Loncat kalender (hari) cand_date -> aligned:",
      (news['aligned_trading_date'] - news['cand_date']).dt.days.value_counts().sort_index().to_dict())
if 'date_raw_dt_wib' in news.columns:
    gap_h = (news['ts_wib'] - news['date_raw_dt_wib']).dt.total_seconds() / 3600
    print("ts_wib (published_raw) - date_raw_dt_wib, dalam jam:"); print(gap_h.describe().round(2).to_string())

ts_wib NaT (dibuang): 0 | di luar rentang BI (dibuang): 3 | artikel tersisa: 4332
Digeser karena lewat 15:00:00: 47.4%
Hari aligned (0=Senin): {0: 1234, 1: 759, 2: 880, 3: 733, 4: 726}
Loncat kalender (hari) cand_date -> aligned: {0: 3313, 1: 344, 2: 455, 3: 88, 4: 49, 5: 21, 6: 27, 7: 22, 8: 1, 9: 2, 10: 7, 11: 3}
ts_wib (published_raw) - date_raw_dt_wib, dalam jam:
count    4332.00
mean       -6.99
std         0.49
min       -25.55
25%        -7.00
50%        -7.00
75%        -7.00
max         8.51


In [27]:
# ===== Agregasi per aligned_trading_date =====
K = 2   # peredam untuk hari dengan sedikit berita

# hitungan kata LM per artikel (sudah setelah flip)
tok = news['lm_text_n_tokens']
news['lm_pos_cnt'] = (news['lm_text_pos'] * tok).round()
news['lm_neg_cnt'] = (news['lm_text_neg'] * tok).round()
news['lm_unc_cnt'] = (news['lm_text_unc'] * tok).round()

g = news.groupby('aligned_trading_date')
daily = g.agg(
    news_count   =('title', 'size'),
    share_shifted=('shifted', 'mean'),
    us_ratio     =('is_us', 'mean'),
    # VADER (judul, sudah di-flip)
    vader_mean   =('vader_title', 'mean'),
    vader_min    =('vader_title', 'min'),
    vader_max    =('vader_title', 'max'),
    vader_std    =('vader_title', 'std'),
    vader_pol_sum=('vader_title_polarity', 'sum'),
    vader_pos_n  =('vader_title_polarity', lambda x: (x > 0).sum()),
    vader_neg_n  =('vader_title_polarity', lambda x: (x < 0).sum()),
    # LM (lower_text, sudah di-flip): polaritas per artikel + jumlah untuk pooling
    si_lm_art    =('lm_text_polarity', 'mean'),
    lm_pos_sum   =('lm_pos_cnt', 'sum'),
    lm_neg_sum   =('lm_neg_cnt', 'sum'),
    lm_unc_sum   =('lm_unc_cnt', 'sum'),
    lm_tok_sum   =('lm_text_n_tokens', 'sum'),
).reset_index()

N = daily['news_count']
daily['vader_pos_ratio'] = daily['vader_pos_n'] / N
daily['vader_neg_ratio'] = daily['vader_neg_n'] / N
daily['si_vader']        = daily['vader_pol_sum'] / N                                   # (n_pos - n_neg) / N
daily['lm_neg_w']        = daily['lm_neg_sum'] / daily['lm_tok_sum']                    # rasio dipool per token
daily['lm_pos_w']        = daily['lm_pos_sum'] / daily['lm_tok_sum']
daily['lm_unc_w']        = daily['lm_unc_sum'] / daily['lm_tok_sum']
daily['lm_net_w']        = (daily['lm_pos_sum'] - daily['lm_neg_sum']) / daily['lm_tok_sum']
daily['lm_polarity_w']   = ((daily['lm_pos_sum'] - daily['lm_neg_sum'])
                            / (daily['lm_pos_sum'] + daily['lm_neg_sum']).replace(0, np.nan))
for c in ['si_vader', 'si_lm_art']:
    daily[f'{c}_damp'] = daily[c] * N / (N + K)

# ---------- Merge ke kalender BI (satu baris per hari bursa) ----------
df = bi.merge(daily, how='left', left_on='Tanggal_dt', right_on='aligned_trading_date',
              validate='one_to_one').drop(columns='aligned_trading_date')
df = df.sort_values('Tanggal_dt').reset_index(drop=True)

df['news_count'] = df['news_count'].fillna(0).astype(int)
df['has_news']   = (df['news_count'] > 0).astype(int)
df['gap_days']   = df['Tanggal_dt'].diff().dt.days.fillna(1).astype(int)   # selisih kalender dari hari bursa sebelumnya
df['news_per_cal_day'] = df['news_count'] / df['gap_days']

FILL0 = [c for c in df.columns if c.startswith(('vader_', 'lm_', 'si_')) or c in ('share_shifted', 'us_ratio')]
df[FILL0] = df[FILL0].fillna(0.0)           # hari tanpa berita: netral, dibedakan oleh has_news / news_count

# ---------- Rolling (kausal) dari jumlah, bukan rata-rata rata-rata ----------
def roll_ratio(num, den, w):
    n_, d_ = df[num].rolling(w, min_periods=1).sum(), df[den].rolling(w, min_periods=1).sum()
    return (n_ / d_.replace(0, np.nan)).fillna(0.0)

for w in (3, 5):
    df[f'si_vader_r{w}']   = roll_ratio('vader_pol_sum', 'news_count', w)
    df[f'lm_net_w_r{w}']   = roll_ratio('lm_pos_sum', 'lm_tok_sum', w) - roll_ratio('lm_neg_sum', 'lm_tok_sum', w)
    df[f'news_count_r{w}'] = df['news_count'].rolling(w, min_periods=1).sum()

# ---------- Cek ----------
assert df['news_count'].sum() == len(news), "ada artikel yang hilang saat merge ke BI"
print(f"Hari bursa: {len(df)} | tanpa berita: {(df['has_news'] == 0).sum()} ({(df['has_news'] == 0).mean():.1%})")
print("Rata-rata news_count per gap_days:"); print(df.groupby('gap_days')['news_count'].agg(['mean', 'count']).round(2).to_string())
print("\n5 hari dengan berita terbanyak (harus Senin/pasca-libur, bukan bug):")
display(df.nlargest(5, 'news_count')[['Tanggal_dt', 'gap_days', 'news_count', 'si_vader', 'lm_net_w']])
print("Korelasi si_vader vs lm_net_w:", round(df.loc[df['has_news'] == 1, ['si_vader', 'lm_net_w']].corr().iloc[0, 1], 3))

Hari bursa: 1202 | tanpa berita: 135 (11.2%)
Rata-rata news_count per gap_days:
           mean  count
gap_days              
1          2.89    923
2          5.22     18
3          4.83    223
4         10.33     18
5          9.92     13
6          3.00      2
8         47.50      2
11        27.00      2
12        26.00      1

5 hari dengan berita terbanyak (harus Senin/pasca-libur, bukan bug):


,Tanggal_dt,gap_days,news_count,si_vader,lm_net_w
1097,2026-03-25,8,80,-0.425000,-0.017411
635,2024-04-16,11,39,-0.282051,-0.019917
914,2025-06-23,3,29,-0.586207,-0.020587
126,2022-03-01,4,28,0.071429,-0.019323
868,2025-04-08,12,26,-0.230769,-0.018091


Korelasi si_vader vs lm_net_w: 0.308


In [28]:
# ===== Bobot waktu + teks harian (TF-IDF dari teks asli) =====
TAU_H = 24.0   # peluruhan: artikel yang lebih tua dari cutoff hari target berbobot lebih kecil

info_time = news['aligned_trading_date'] + pd.Timedelta(hours=15)            # cutoff hari bursa target
news['age_h']   = ((info_time - news['ts_wib']).dt.total_seconds() / 3600).clip(lower=0)
news['w_time']  = np.exp(-news['age_h'] / TAU_H)
news['w_vader'] = news['w_time'] * news['vader_title_polarity']
news['w_lmnet'] = news['w_time'] * news['lm_text_net']

g2 = news.groupby('aligned_trading_date')
wsum = g2['w_time'].sum()
extra = pd.DataFrame({
    'w_sum':          wsum,                                   # "volume segar": jumlah bobot waktu
    'si_vader_tw':    g2['w_vader'].sum() / wsum,             # rata-rata berbobot waktu
    'lm_net_tw':      g2['w_lmnet'].sum() / wsum,
    'all_lower_text': g2['lower_text'].agg(' '.join),         # teks asli (lowercase), bukan lemmatized
}).reset_index().rename(columns={'aligned_trading_date': 'Tanggal_dt'})

df = df.drop(columns=[c for c in extra.columns if c != 'Tanggal_dt' and c in df.columns])
df = df.merge(extra, on='Tanggal_dt', how='left', validate='one_to_one')
df['all_lower_text'] = df['all_lower_text'].fillna('')
df[['w_sum', 'si_vader_tw', 'lm_net_tw']] = df[['w_sum', 'si_vader_tw', 'lm_net_tw']].fillna(0.0)

print("age_h (jam) median/p90/max:", news['age_h'].quantile([.5, .9, 1]).round(1).tolist())
print("Bobot rata-rata per gap_days:"); print(df.groupby('gap_days')['w_sum'].mean().round(2).to_string())

age_h (jam) median/p90/max: [17.8, 66.7, 287.3]
Bobot rata-rata per gap_days:
gap_days
1     1.84
2     2.38
3     1.80
4     3.10
5     1.99
6     0.12
8     9.49
11    2.63
12    4.25


# Modelling: Prediksi Return USD/IDR (Regresi)

- **Target**: log return harian kurs tengah BI (%), digeser `H = 1` hari ke depan. Nilai positif berarti rupiah melemah.
- **Fitur** hanya memakai informasi sampai cutoff 15:00 hari t.
- **Split kronologis** 70/15/15, tanpa shuffle. Scaler dan TF-IDF di-fit hanya pada data latih.
- **Ablation**: AR, +volume, +VADER, +LM, +VADER+LM, +bobot waktu, +TF-IDF (teks asli). Semua dibandingkan dengan baseline naive (prediksi 0).

In [29]:
# ===== Target, fitur, split kronologis =====
H = 1   # prediksi return hari t+H dari info sampai t (cutoff 15:00 hari t). H=0 hanya jika kurs BI ditetapkan setelah cutoff.

df['mid'] = (df['Kurs Jual'] + df['Kurs Beli']) / 2
df['ret'] = 100 * np.log(df['mid']).diff()          # log return harian (%), + = rupiah melemah
df['y']   = df['ret'].shift(-H)

for k in range(5):
    df[f'ret_lag{k}'] = df['ret'].shift(k)          # lag0 = ret_t, sudah diketahui di t
df['vol_5']  = df['ret'].rolling(5).std()
df['vol_20'] = df['ret'].rolling(20).std()
df['mom_5']  = df['ret'].rolling(5).sum()

AR_COLS    = [f'ret_lag{k}' for k in range(5)] + ['vol_5', 'vol_20', 'mom_5']
VOL_COLS   = ['news_count', 'has_news', 'gap_days', 'news_per_cal_day', 'news_count_r3', 'news_count_r5',
              'share_shifted', 'us_ratio']
VADER_COLS = ['si_vader', 'vader_mean', 'vader_min', 'vader_max', 'vader_std',
              'vader_pos_ratio', 'vader_neg_ratio', 'si_vader_r3', 'si_vader_r5']
LM_COLS    = ['lm_neg_w', 'lm_pos_w', 'lm_unc_w', 'lm_net_w', 'lm_polarity_w', 'si_lm_art',
              'lm_net_w_r3', 'lm_net_w_r5']
TW_COLS    = ['w_sum', 'si_vader_tw', 'lm_net_tw']

ALL_F = AR_COLS + VOL_COLS + VADER_COLS + LM_COLS + TW_COLS
ok = df[ALL_F].notna().all(axis=1) & df['y'].notna()      # buang baris awal (rolling) & baris terakhir (tanpa target)
ds = df[ok].reset_index(drop=True)

n = len(ds); i1, i2 = int(n * 0.70), int(n * 0.85)
tr, va, te = np.arange(0, i1), np.arange(i1, i2), np.arange(i2, n)
trva = np.concatenate([tr, va])
for name, ix in [('Train', tr), ('Val', va), ('Test', te)]:
    d = ds.loc[ix]
    print(f"{name:5}: {len(ix)} | {d['Tanggal_dt'].min().date()} -> {d['Tanggal_dt'].max().date()}"
          f" | news/hari {d['news_count'].mean():.2f} | tanpa berita {(d['has_news'] == 0).mean():.0%} | std y {d['y'].std():.3f}")

Train: 826 | 2021-09-29 -> 2025-02-25 | news/hari 3.00 | tanpa berita 13% | std y 0.344
Val  : 177 | 2025-02-26 -> 2025-11-25 | news/hari 3.47 | tanpa berita 10% | std y 0.340
Test : 178 | 2025-11-26 -> 2026-08-31 | news/hari 6.83 | tanpa berita 2% | std y 0.290


In [30]:
# ===== Model regresi (Ridge) + ablation =====
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

STOP = sorted(set(stopwords.words('english')) | set(globals().get('CUSTOM_STOPWORDS', set())))

def fit_predict(fit_ix, pred_ix, cols, use_text, alpha, return_model=False):
    sc = StandardScaler().fit(ds.loc[fit_ix, cols])                       # scaler & TF-IDF di-fit hanya pada data fit
    Xf, Xp = sc.transform(ds.loc[fit_ix, cols]), sc.transform(ds.loc[pred_ix, cols])
    tf = None
    if use_text:
        tf = TfidfVectorizer(stop_words=STOP, max_features=2000, ngram_range=(1, 2), min_df=5, sublinear_tf=True)
        Tf = tf.fit_transform(ds.loc[fit_ix, 'all_lower_text'])           # teks asli (lowercase), bukan lemmatized
        Tp = tf.transform(ds.loc[pred_ix, 'all_lower_text'])
        Xf, Xp = hstack([csr_matrix(Xf), Tf]).tocsr(), hstack([csr_matrix(Xp), Tp]).tocsr()
    m = Ridge(alpha=alpha).fit(Xf, ds.loc[fit_ix, 'y'])
    p = m.predict(Xp)
    return (p, m, sc, tf) if return_model else p

def evaluate(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {'RMSE': np.sqrt(mean_squared_error(y, p)), 'MAE': mean_absolute_error(y, p), 'R2': r2_score(y, p),
            'R2_oos_vs_zero': 1 - ((y - p) ** 2).sum() / (y ** 2).sum(),      # > 0 = lebih baik dari prediksi 0
            'DirAcc': np.mean(np.sign(y) == np.sign(p)) if np.any(p) else np.nan}   # naive (semua 0): tidak berlaku

EXPS = {
    '1 AR':                       (AR_COLS, False),
    '2 AR+VOL':                   (AR_COLS + VOL_COLS, False),
    '3 AR+VOL+VADER':             (AR_COLS + VOL_COLS + VADER_COLS, False),
    '4 AR+VOL+LM':                (AR_COLS + VOL_COLS + LM_COLS, False),
    '5 AR+VOL+VADER+LM':          (AR_COLS + VOL_COLS + VADER_COLS + LM_COLS, False),
    '6 +bobot waktu':             (AR_COLS + VOL_COLS + VADER_COLS + LM_COLS + TW_COLS, False),
    '7 +TF-IDF (teks asli)':      (AR_COLS + VOL_COLS + VADER_COLS + LM_COLS, True),
}
ALPHAS = [1, 10, 100, 1e3, 1e4, 1e5]
y_va, y_te = ds.loc[va, 'y'], ds.loc[te, 'y']

rows_val = [{'model': '0 naive (prediksi 0)', **evaluate(y_va, np.zeros(len(va)))}]
rows_te  = [{'model': '0 naive (prediksi 0)', **evaluate(y_te, np.zeros(len(te)))}]
best = {}
for name, (cols, use_text) in EXPS.items():
    sc_ = {a: evaluate(y_va, fit_predict(tr, va, cols, use_text, a)) for a in ALPHAS}
    a_b = min(sc_, key=lambda a: sc_[a]['RMSE'])          # alpha dipilih di validation
    best[name] = a_b
    rows_val.append({'model': f'{name} [a={a_b:g}]', **sc_[a_b]})
    rows_te.append({'model': f'{name} [a={a_b:g}]', **evaluate(y_te, fit_predict(trva, te, cols, use_text, a_b))})

pd.set_option('display.float_format', '{:.4f}'.format)
print('VALIDATION'); display(pd.DataFrame(rows_val).set_index('model'))
print('TEST (refit train+val)'); display(pd.DataFrame(rows_te).set_index('model'))

VALIDATION


,RMSE,MAE,R2,R2_oos_vs_zero,DirAcc
model,,,,,
0 naive (prediksi 0),0.3391,0.2484,-0.0013,0.0000,NaN
1 AR [a=1000],0.3376,0.2496,0.0077,0.0090,0.5198
2 AR+VOL [a=1],0.3342,0.2550,0.0276,0.0289,0.4915
3 AR+VOL+VADER [a=100],0.3355,0.2556,0.0201,0.0214,0.5085
4 AR+VOL+LM [a=1],0.3374,0.2580,0.0088,0.0100,0.4802
5 AR+VOL+VADER+LM [a=1000],0.3382,0.2533,0.0041,0.0054,0.5141
6 +bobot waktu [a=1000],0.3386,0.2534,0.0018,0.0030,0.5141
7 +TF-IDF (teks asli) [a=1000],0.3382,0.2532,0.0044,0.0056,0.5141


TEST (refit train+val)


,RMSE,MAE,R2,R2_oos_vs_zero,DirAcc
model,,,,,
0 naive (prediksi 0),0.2918,0.2212,-0.0148,0.0000,NaN
1 AR [a=1000],0.2892,0.2176,0.0030,0.0175,0.5787
2 AR+VOL [a=1],0.3167,0.2253,-0.1959,-0.1785,0.5056
3 AR+VOL+VADER [a=100],0.3026,0.2208,-0.0919,-0.0759,0.5281
4 AR+VOL+LM [a=1],0.3179,0.2250,-0.2045,-0.1869,0.5225
5 AR+VOL+VADER+LM [a=1000],0.2915,0.2167,-0.0128,0.0020,0.5506
6 +bobot waktu [a=1000],0.2915,0.2166,-0.0128,0.0020,0.5449
7 +TF-IDF (teks asli) [a=1000],0.2915,0.2167,-0.0129,0.0018,0.5506


In [31]:
# ===== Bobot fitur (koefisien terstandarisasi) =====
name = '6 +bobot waktu'
cols, use_text = EXPS[name]
_, m, sc, _ = fit_predict(trva, te, cols, use_text, best[name], return_model=True)
coef = pd.Series(m.coef_, index=cols).sort_values(key=np.abs, ascending=False)
print(f'{name}: 15 koefisien terbesar (fitur terstandarisasi; + = rupiah melemah esok hari)')
display(coef.head(15).round(4).to_frame('coef'))

name = '7 +TF-IDF (teks asli)'
cols, use_text = EXPS[name]
_, m, sc, tf = fit_predict(trva, te, cols, use_text, best[name], return_model=True)
terms, w = np.array(tf.get_feature_names_out()), m.coef_[len(cols):]
o = np.argsort(w)
print('Rupiah melemah:', list(terms[o[-12:]][::-1]))
print('Rupiah menguat:', list(terms[o[:12]]))

6 +bobot waktu: 15 koefisien terbesar (fitur terstandarisasi; + = rupiah melemah esok hari)


,coef
ret_lag1,-0.0193
ret_lag0,0.0187
gap_days,0.0155
news_count,0.0143
vader_std,-0.0059
ret_lag3,0.0057
si_vader_r5,-0.0056
has_news,-0.0051
lm_net_w_r5,0.0047
news_per_cal_day,-0.0045


Rupiah melemah: ['industrial', 'russian', 'western', 'delhi', 'jones', 'sure', '300', 'able', 'stimulus', 'consumers', 'export', 'opec']
Rupiah menguat: ['documents', 'net', 'call', 'share', 'uae', 'emirates', 'earnings', 'arab', 'first quarter', 'crypto', 'investing', 'arab emirates']


In [32]:
# ===== Simpan tabel harian untuk laporan =====
os.makedirs('data/processed', exist_ok=True)
df.drop(columns=['all_lower_text']).to_csv('data/processed/daily_features.csv', index=False)
df[['Tanggal_dt', 'news_count', 'all_lower_text']].to_csv('data/processed/daily_text.csv', index=False)
print("Tersimpan:", df.shape, "-> data/processed/")

Tersimpan: (1202, 57) -> data/processed/
